# 04. 모델 선택 (Model Selection)

**Why (왜 이 노트북이 필요한가)**: `03_features.ipynb`에서 888개 컬럼짜리 피처를 다 만들었지만, 아직 어떤 모델로 예측할지는 정하지 않았습니다. CLAUDE.md 로드맵 7단계 — 후보 모델(선형회귀/LightGBM/XGBoost/CatBoost)을 **공정한 조건**에서 비교해서 `05_tuning.ipynb`로 넘길 1~2개를 고릅니다.

**입력**: `data/processed/train_features_v1.parquet` (26304행 × 888열)
**출력**: 이 노트북은 parquet을 새로 만들지 않습니다. 결과는 비교 표(마크다운)로 남기고, 최종 정리는 `reports/04_model_selection.md`에 기록합니다.

## 이 노트북에서 반드시 지키는 것

1. **A안 + B안 항상 병행** (`timeseries-validation` 스킬, CLAUDE.md 5장 확정 사항): 모든 비교표에 A안(2022~2023 학습→2024 검증)과 B안(3-fold 확장 윈도우) 점수를 같이 적습니다.
2. **fold-safe 피처만 사용**: `{group}_ws_est`, `{group}_power_curve_est`는 2022~2024 **전체** 데이터로 학습된 값이라 그대로 쓰면 미래 정보가 회귀계수를 통해 새어 들어갑니다(예측기준시점 원칙 위반). `03_features.ipynb`가 이미 만들어둔 `_cv_2023_07/2024_01/2024_07` fold-safe 버전을 씁니다.
3. **새로 발견한 것 (이번 세션)**: `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`도 전부 fold-safe 하지 않은 `ws_est`에서 파생됐습니다. 이 노트북 2절에서 각 fold의 `ws_est_cv_*`로부터 **그 자리에서 다시 계산**해서 씁니다(민석님 확인 후 채택한 방식 — "폴드별 즉석 재계산").


## 1. 셋업

이 셀은 파이썬 실행 환경(venv인지 확인), 필요한 라이브러리, 경로 상수, 시드를 준비합니다.

이번 노트북부터 **LightGBM / XGBoost / CatBoost / scikit-learn**을 새로 씁니다 (venv에 설치 완료: `lightgbm 4.7.0`, `xgboost 3.3.0`, `catboost 1.2.10`, `scikit-learn 1.9.0`). `requirements.txt`는 민석님과 상의한 대로 튜닝 단계까지 패키지가 다 들어간 뒤 한 번에 정리합니다.

In [1]:
import sys
print(sys.executable)  # venv\Scripts\python.exe 인지 확인

c:\Users\cho03\Desktop\wind_forecast_new\venv\Scripts\python.exe


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

PROCESSED_DIR = Path("../data/processed") if Path.cwd().name == "notebooks" else Path("data/processed")
GROUP_COLS = TARGET_COLS  # ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
print(CAPACITY_KWH)

{'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000}


## 2. 데이터 로드

`train_features_v1.parquet`만 로드합니다. `test_features_v1.parquet`은 최종 재학습(9~10단계, `ensemble-final` 스킬)에서 씁니다 — 지금은 모델 비교 단계라 미래(2025) 예측은 하지 않습니다.

In [3]:
train = pd.read_parquet(PROCESSED_DIR / "train_features_v1.parquet")
print(train.shape)
display(train[["kst_dtm", *GROUP_COLS]].head(3))

(26304, 888)


,kst_dtm,kpx_group_1,kpx_group_2,kpx_group_3
0,2022-01-01 01:00:00,12004.421,9719.242,NaN
1,2022-01-01 02:00:00,12901.137,10297.768,NaN
2,2022-01-01 03:00:00,12091.200,10731.663,NaN


**확인할 것**: `(26304, 888)`. `kst_dtm`이 시간순으로 정렬돼 있는지, `kpx_group_1/2/3` 값이 정상 범위(0~설비용량 근처)인지 육안 확인.

In [4]:
print("kst_dtm 범위:", train["kst_dtm"].min(), "~", train["kst_dtm"].max())
print("결측 라벨 수:")
for g in GROUP_COLS:
    print(f"  {g}: {train[g].isna().sum()}건 (전체 {len(train)}행 중)")

kst_dtm 범위: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
결측 라벨 수:
  kpx_group_1: 516건 (전체 26304행 중)
  kpx_group_2: 516건 (전체 26304행 중)
  kpx_group_3: 9201건 (전체 26304행 중)


**확인할 것**: `kpx_group_3`은 2022년치가 전부 NaN이라 다른 그룹보다 결측이 많은 게 정상입니다(02_eda에서 이미 확인된 사실 — group_3은 2023년부터 라벨 존재).

## 3. 모델 입력 피처 정의 — 공통 피처 / 그룹 전용 피처 / 제외할 피처

**결론부터**: 888개 컬럼을 세 종류로 나눕니다.

1. **제외 컬럼** (`NON_FEATURE_COLS`): 정답(`kpx_group_1/2/3`), 시각(`kst_dtm`), 예보 도착 시각(`data_available_kst_dtm_*`), 검증용 임시 컬럼(`year`), 그리고 **SCADA 실측값**(`scada_ws_kpx_group_*`, `scada_kpx_group_*`) — SCADA는 test에 없는 학습 전용 정답 보조자료라 모델 입력으로 쓰면 안 됩니다(`03_features.ipynb` 요약에서 이미 못박은 규칙).
2. **공통 피처** (`COMMON_RAW_COLS`): 그룹 이름이 안 붙은 컬럼 — GFS 9개 격자, LDAPS 16격자 평균, `hour_sin/cos`, `month_sin/cos` 등. 세 그룹 모델 모두에 공통으로 씁니다.
3. **그룹 전용 피처** (`GROUP_SPECIFIC_COLS[g]`): `{group}_`로 시작하는 컬럼 — 그 그룹의 최근접 격자, 추정풍속, 파워커브 추정치, 결빙·돌풍 위험 등. 이 그룹의 모델에만 씁니다.

In [5]:
NON_FEATURE_COLS = set([
    "kst_dtm",
    "data_available_kst_dtm_gfs",
    "data_available_kst_dtm_ldaps",
    "year",
    *TARGET_COLS,
    *[f"scada_ws_{g}" for g in GROUP_COLS],
    *[f"scada_{g}" for g in GROUP_COLS],
])

COMMON_RAW_COLS = [
    c for c in train.columns
    if c not in NON_FEATURE_COLS and not any(c.startswith(f"{g}_") for g in GROUP_COLS)
]
GROUP_SPECIFIC_COLS = {
    g: [c for c in train.columns if c.startswith(f"{g}_") and c not in NON_FEATURE_COLS]
    for g in GROUP_COLS
}

print("제외 컬럼 수:", len(NON_FEATURE_COLS))
print("공통 피처 수:", len(COMMON_RAW_COLS))
for g in GROUP_COLS:
    print(f"{g} 전용 피처 수:", len(GROUP_SPECIFIC_COLS[g]))

제외 컬럼 수: 13
공통 피처 수: 804
kpx_group_1 전용 피처 수: 24
kpx_group_2 전용 피처 수: 24
kpx_group_3 전용 피처 수: 23


**확인할 것**: 제외 컬럼 수 + 공통 피처 수 + 그룹 전용 피처 수(3개 그룹 합) = 888이 돼야 합니다(빠뜨리거나 중복된 컬럼이 없는지 검산).

In [6]:
total = len(NON_FEATURE_COLS) + len(COMMON_RAW_COLS) + sum(len(v) for v in GROUP_SPECIFIC_COLS.values())
print(total, "== 888 이어야 함:", total == train.shape[1])

888 == 888 이어야 함: True


## 4. 검증 fold 정의 — A안 + B안

**무엇을**: A안(2022~2023 학습 → 2024 전체 검증)과 B안(확장 윈도우 3-fold)의 학습/검증 구간 경계를 정의합니다.

**왜 항상 같이 보는가**: `timeseries-validation` 스킬 1절 — A안 하나만 보면 2024년이 이례적인 해였을 경우 속을 수 있고, B안 fold만 보면 fold별 6개월 구간이 계절에 치우칠 수 있습니다. 두 방식을 같이 봐야 서로의 편향을 잡아줍니다.

| fold | 학습 구간 | 검증 구간 | 쓰는 fold-safe 컬럼 |
|---|---|---|---|
| A안(2024) | 2022-01 ~ 2023-12 | 2024-01 ~ 2024-12 (전체) | `*_cv_2024_01` |
| B안 fold1 | ~2023-06 | 2023-07 ~ 2023-12 | `*_cv_2023_07` |
| B안 fold2 | ~2023-12 | 2024-01 ~ 2024-06 | `*_cv_2024_01` |
| B안 fold3 | ~2024-06 | 2024-07 ~ 2024-12 | `*_cv_2024_07` |

**A안과 B안 fold2가 같은 cv 컬럼을 쓰는 이유**: 둘 다 학습 구간이 정확히 2022~2023년 전체로 같기 때문입니다. `ws_est_cv_2024_01`은 "2024-01-01 이전 데이터로만 학습한 버전"이라는 뜻이라, 검증 구간이 2024년 상반기(B안 fold2)든 2024년 전체(A안)든 상관없이 안전하게 재사용할 수 있습니다 — cv 컬럼의 안전성은 **학습에 쓰인 기간**으로만 결정되지, 검증 기간의 길이와는 무관하기 때문입니다.

In [7]:
CUTOFFS = {
    "2023_07": pd.Timestamp("2023-07-01"),
    "2024_01": pd.Timestamp("2024-01-01"),
    "2024_07": pd.Timestamp("2024-07-01"),
}
END_TRAIN = pd.Timestamp("2025-01-01")  # train 데이터의 끝(다음 시각의 시작점 기준, 배타적 상한)

FOLD_SPECS = {
    "A안(2024)":  dict(cv_suffix="2024_01", valid_start=CUTOFFS["2024_01"], valid_end=END_TRAIN),
    "B안 fold1": dict(cv_suffix="2023_07", valid_start=CUTOFFS["2023_07"], valid_end=CUTOFFS["2024_01"]),
    "B안 fold2": dict(cv_suffix="2024_01", valid_start=CUTOFFS["2024_01"], valid_end=CUTOFFS["2024_07"]),
    "B안 fold3": dict(cv_suffix="2024_07", valid_start=CUTOFFS["2024_07"], valid_end=END_TRAIN),
}

for name, spec in FOLD_SPECS.items():
    train_mask = train["kst_dtm"] < CUTOFFS[spec["cv_suffix"]]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    print(f"{name}: 학습 후보 {train_mask.sum()}행 / 검증 후보 {valid_mask.sum()}행")
    for g in GROUP_COLS:
        n_tr = (train_mask & train[g].notna()).sum()
        n_va = (valid_mask & train[g].notna()).sum()
        print(f"   {g}: 학습 라벨 {n_tr}행 / 검증 라벨 {n_va}행")

A안(2024): 학습 후보 17519행 / 검증 후보 8784행
   kpx_group_1: 학습 라벨 17341행 / 검증 라벨 8446행
   kpx_group_2: 학습 라벨 17341행 / 검증 라벨 8446행
   kpx_group_3: 학습 라벨 8662행 / 검증 라벨 8440행
B안 fold1: 학습 후보 13103행 / 검증 후보 4416행
   kpx_group_1: 학습 라벨 12925행 / 검증 라벨 4416행
   kpx_group_2: 학습 라벨 12925행 / 검증 라벨 4416행
   kpx_group_3: 학습 라벨 4246행 / 검증 라벨 4416행
B안 fold2: 학습 후보 17519행 / 검증 후보 4368행
   kpx_group_1: 학습 라벨 17341행 / 검증 라벨 4030행
   kpx_group_2: 학습 라벨 17341행 / 검증 라벨 4030행
   kpx_group_3: 학습 라벨 8662행 / 검증 라벨 4030행
B안 fold3: 학습 후보 21887행 / 검증 후보 4416행
   kpx_group_1: 학습 라벨 21371행 / 검증 라벨 4416행
   kpx_group_2: 학습 라벨 21371행 / 검증 라벨 4416행
   kpx_group_3: 학습 라벨 12692행 / 검증 라벨 4410행


**확인할 것**: `kpx_group_3`은 2023년부터 라벨이 있으므로, B안 fold1(학습 구간이 ~2023-06)의 `group_3` 학습 라벨 수가 다른 그룹보다 훨씬 적게 나오는 게 정상입니다(2023년 상반기치만 있음). 이 부분은 `timeseries-validation` 스킬에서 이미 알려진 제약이라 결과 해석 시 감안해야 합니다.

## 5. fold-safe 파생 피처 재계산 (이번 세션에 새로 발견한 부분)

**무엇을**: `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`를 원래 `ws_est`(전체 데이터 학습판) 대신 **각 fold의 `ws_est_cv_{cutoff}`로부터 그 자리에서 재계산**합니다.

**왜**: `03_features.ipynb`를 다시 읽어보니 이 6종 피처가 전부 fold-safe하지 않은 `ws_est`를 통과시켜 만들어져 있었습니다(`ws_est` 본체와 `power_curve_est`만 fold-safe `_cv_*` 버전이 있었음). 그대로 A/B 검증에 쓰면 예측기준시점 원칙을 어기게 됩니다. 다행히 전부 **고정 임계값(3.0/12.0/15.0 m/s)이나 정해진 공식**으로 만든 파생값이라, 학습(fit)이 필요 없이 산술 연산만 다시 하면 fold-safe 버전을 만들 수 있습니다.

**공식은 `03_features.ipynb`와 완전히 동일하게 재사용**(임계값을 새로 정하지 않음 — 이미 근거를 남긴 값 그대로):
- `ws_est_corrected = ws × (공기밀도/표준밀도)^(1/3)` (4절), `sq`/`cube`는 이 보정값의 제곱/세제곱 (03_features 3-2절 순서 그대로: sq/cube는 corrected 기준)
- `regime_calm/ramp/rated`: `ws < 3.0` / `3.0 ≤ ws < 12.0` / `ws ≥ 12.0` (13-1절)
- `high_wind_caution`: `ws ≥ 15.0` (13-2절)
- `icing_risk`(group_1/2 전용): 기온 < 0℃ AND `3.0 ≤ ws ≤ 7.0` (10절)

`{group}_air_density`는 LDAPS 기온·기압만으로 계산돼 있어(라벨/SCADA 의존 없음) 그대로 재사용해도 안전합니다.

In [8]:
GROUP_NEAREST_LDAPS = {"kpx_group_1": 5, "kpx_group_2": 6, "kpx_group_3": 12}
ICING_RISK_GROUPS = ["kpx_group_1", "kpx_group_2"]

CUT_IN, RATED, HIGH_WIND_THRESHOLD = 3.0, 12.0, 15.0
ICING_WS_MIN, ICING_WS_MAX = 3.0, 7.0
R_DRY_AIR = 287.05
RHO_STANDARD = 1.225


def add_fold_safe_ws_features(df, group_col, ws_col):
    """ws_col(그 fold에서 안전한 풍속 추정치)로부터 corrected/sq/cube/regime_*/high_wind_caution/icing_risk를
    그 자리에서 다시 계산한다. 공식·임계값은 03_features.ipynb 4/10/13절과 완전히 동일하다."""
    ws = df[ws_col]
    rho = df[f"{group_col}_air_density"]
    corrected = ws * (rho / RHO_STANDARD) ** (1 / 3)

    out = {}
    out[f"{ws_col}__corrected"] = corrected
    out[f"{ws_col}__sq"] = corrected ** 2
    out[f"{ws_col}__cube"] = corrected ** 3
    out[f"{ws_col}__regime_calm"] = (ws < CUT_IN).astype(int)
    out[f"{ws_col}__regime_ramp"] = ((ws >= CUT_IN) & (ws < RATED)).astype(int)
    out[f"{ws_col}__regime_rated"] = (ws >= RATED).astype(int)
    out[f"{ws_col}__high_wind_caution"] = (ws >= HIGH_WIND_THRESHOLD).astype(int)

    if group_col in ICING_RISK_GROUPS:
        grid_id = GROUP_NEAREST_LDAPS[group_col]
        temp_col = f"ldaps_g{grid_id}_heightAboveGround_2_t"
        is_freezing = df[temp_col] < 273.15
        in_range = ws.between(ICING_WS_MIN, ICING_WS_MAX)
        out[f"{ws_col}__icing_risk"] = (is_freezing & in_range).astype(int)

    return pd.DataFrame(out, index=df.index)

In [9]:
# 확인: A안 fold(ws_est_cv_2024_01)로 만든 fold-safe 파생값이 원본 03_features 파생값과 "비슷한 크기"인지 산점 비교
sample_g = "kpx_group_1"
fs_sample = add_fold_safe_ws_features(train, sample_g, f"{sample_g}_ws_est_cv_2024_01")
print("재계산본 regime 분포:")
print(fs_sample[f"{sample_g}_ws_est_cv_2024_01__regime_calm"].value_counts(normalize=True).round(4))
print()
print("원본(03_features, 전체데이터판) regime 분포 (참고용, 실제 학습엔 fold-safe본만 사용):")
print(train[f"{sample_g}_regime_calm"].value_counts(normalize=True).round(4))

재계산본 regime 분포:
kpx_group_1_ws_est_cv_2024_01__regime_calm
0    0.9797
1    0.0203
Name: proportion, dtype: float64

원본(03_features, 전체데이터판) regime 분포 (참고용, 실제 학습엔 fold-safe본만 사용):
kpx_group_1_regime_calm
0    0.981
1    0.019
Name: proportion, dtype: float64


**확인할 것**: 두 분포(재계산본 vs 원본)가 완전히 같지는 않아도(학습 데이터 범위가 다르므로) 비슷한 자릿수로 나오면 정상입니다. 극단적으로 다르면(예: 한쪽은 90%, 한쪽은 1%) 공식을 잘못 옮겼다는 신호이니 코드를 다시 확인해야 합니다.

## 6. 그룹별 모델 입력 프레임 빌더 + 채점 함수

**무엇을**: 지금까지 정의한 것들을 묶어서, "그룹 g, fold X" 조합마다 모델에 넣을 피처 테이블을 한 번에 만들어주는 함수(`build_group_feature_frame`)와, 대회 공식 산식(`src/metric.py`)으로 점수를 매기는 함수를 만듭니다.

**leaky 컬럼 제외 규칙**: `{group}_ws_est`, `{group}_ws_est_sq/_cube/_corrected`, `{group}_regime_calm/ramp/rated`, `{group}_high_wind_caution`, `{group}_icing_risk`, 그리고 **이 fold가 아닌 다른 cutoff의 `_cv_*` 컬럼**은 전부 빼고, 이 fold에 맞는 `ws_est_cv_{cutoff}` / `power_curve_est_cv_{cutoff}`와 5절에서 재계산한 fold-safe 파생값으로 채웁니다.

In [10]:
def all_cv_cols_for_group(g):
    return [c for c in train.columns if c.startswith(f"{g}_ws_est_cv_") or c.startswith(f"{g}_power_curve_est_cv_")]


def leaky_cols_for_group(g):
    base = [f"{g}_ws_est", f"{g}_ws_est_sq", f"{g}_ws_est_cube", f"{g}_ws_est_corrected",
            f"{g}_regime_calm", f"{g}_regime_ramp", f"{g}_regime_rated", f"{g}_high_wind_caution"]
    if g in ICING_RISK_GROUPS:
        base.append(f"{g}_icing_risk")
    return base


def build_group_feature_frame(df, g, cv_suffix):
    """cv_suffix가 주어지면(A/B 검증용) fold-safe 버전 사용.
    cv_suffix=None이면(최종 전체 재학습 전용, 지금 단계에서는 안 씀) 원본 ws_est/power_curve_est 그대로 사용."""
    all_cv = set(all_cv_cols_for_group(g))
    if cv_suffix is None:
        base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in all_cv]
        return df[COMMON_RAW_COLS + base_cols].copy()

    ws_col = f"{g}_ws_est_cv_{cv_suffix}"
    pc_col = f"{g}_power_curve_est_cv_{cv_suffix}"
    exclude = set(leaky_cols_for_group(g)) | (all_cv - {ws_col, pc_col})
    base_cols = [c for c in GROUP_SPECIFIC_COLS[g] if c not in exclude]
    fs = add_fold_safe_ws_features(df, g, ws_col)
    return pd.concat([df[COMMON_RAW_COLS + base_cols], fs], axis=1)


# 안전 점검: NaN 없는지, leaky 컬럼이 실수로 남아있진 않은지
for g in GROUP_COLS:
    for suffix in ["2023_07", "2024_01", "2024_07"]:
        frame = build_group_feature_frame(train, g, suffix)
        assert frame.isna().sum().sum() == 0, f"{g} {suffix}: NaN 발견"
        assert not (set(frame.columns) & set(leaky_cols_for_group(g))), f"{g} {suffix}: leaky 컬럼 잔존"
print("전체 fold x 그룹 조합에서 NaN 없음, leaky 컬럼 잔존 없음 확인")

전체 fold x 그룹 조합에서 NaN 없음, leaky 컬럼 잔존 없음 확인


In [11]:
def score_predictions(actual_df, pred_df):
    """대회 공식 metric()을 그대로 사용 (src/metric.py 수정 금지 원칙)."""
    return metric(actual_df[TARGET_COLS], pred_df[TARGET_COLS])

## 7. 베이스라인 사다리 (`model-selection` 스킬 1절)

복잡한 모델의 개선 효과는 단순한 기준과 비교해야 의미가 있습니다. 아래에서 위로 4단계를 순서대로 쌓습니다.

1. **시간대×월 평균** (기상 정보 전혀 안 씀): 이 점수가 "기상 정보의 가치" 자체를 재는 바닥입니다.
2. **물리 파워커브** (ML 없음): SCADA로 만든 파워커브에 fold-safe 추정풍속만 통과시킨 예측. 도메인 지식만으로 어디까지 가는지 봅니다.
3. **선형회귀** (피처 8개뿐: 추정풍속·제곱·세제곱·풍향 sin/cos·시간 sin/cos·월 sin/cos)
4. **LightGBM 기본값** (876개 전체 피처, 본 게임의 출발점)

각 단계 점수 차이가 "무엇이 성능을 만드는가"를 분해해서 보여줍니다.

### 7-1. 베이스라인 0 — 시간대×월 평균 (기상 미사용)

In [12]:
def baseline_hour_month_mean(g, train_mask, valid_idx):
    train_g = train.loc[train_mask & train[g].notna()]
    profile = train_g.groupby([train_g["kst_dtm"].dt.hour, train_g["kst_dtm"].dt.month])[g].mean()
    overall_mean = train_g[g].mean()
    hm_keys = list(zip(train.loc[valid_idx, "kst_dtm"].dt.hour, train.loc[valid_idx, "kst_dtm"].dt.month))
    pred = pd.Series([profile.get(k, overall_mean) for k in hm_keys], index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 7-2. 베이스라인 1 — 물리 파워커브 (ML 없음)

In [13]:
def baseline_power_curve(g, cv_suffix, valid_idx):
    pc_col = f"{g}_power_curve_est_cv_{cv_suffix}"
    return train.loc[valid_idx, pc_col].clip(lower=0, upper=CAPACITY_KWH[g])

### 7-3. 베이스라인 2 — 선형회귀 (소수 피처: 추정풍속·제곱·세제곱·풍향/시간/월 sin·cos)

In [14]:
def linreg_feature_frame(g, cv_suffix):
    ws_col = f"{g}_ws_est_cv_{cv_suffix}"
    fs = add_fold_safe_ws_features(train, g, ws_col)
    return pd.concat([
        train[[ws_col, f"{g}_wd_sin", f"{g}_wd_cos", "hour_sin", "hour_cos", "month_sin", "month_cos"]],
        fs[[f"{ws_col}__sq", f"{ws_col}__cube"]],
    ], axis=1)


def baseline_linreg(g, cv_suffix, train_mask, valid_idx):
    X_full = linreg_feature_frame(g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    model = LinearRegression()
    model.fit(X_full.loc[fit_idx], train.loc[fit_idx, g])
    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 7-4. 베이스라인 3 — LightGBM 기본값

**early stopping 설계**: 검증 fold의 데이터로 early stopping을 하면(몇 라운드에서 멈출지 결정하는 데 검증 정보를 쓰는 셈이라) 또 다른 형태의 정보 누수가 됩니다. 그래서 **학습 구간 내부**에서 시간순으로 마지막 10%를 떼어 early stopping 전용 검증셋으로 씁니다(공식 검증 fold와는 무관).

In [15]:
def baseline_lgbm(g, cv_suffix, train_mask, valid_idx):
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)  # 학습 구간 내부에서 시간순 마지막 10%를 early stopping용으로 분리
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)

    model = lgb.LGBMRegressor(objective="l1", random_state=SEED, n_estimators=2000, verbosity=-1)
    model.fit(
        X_full.loc[tr_mask], train.loc[tr_mask, g],
        eval_set=[(X_full.loc[es_mask], train.loc[es_mask, g])],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g]), model.best_iteration_

### 7-5. 4개 베이스라인을 모든 fold(A안+B안 3개)에서 실행하고 표로 비교

In [16]:
baseline_results = []
lgbm_best_iters = {}

for fold_name, spec in FOLD_SPECS.items():
    cv_suffix = spec["cv_suffix"]
    train_mask = train["kst_dtm"] < CUTOFFS[cv_suffix]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    valid_idx = train.index[valid_mask]
    actual_df = train.loc[valid_idx, TARGET_COLS]

    preds_by_baseline = {"0_시간x월평균": {}, "1_물리파워커브": {}, "2_선형회귀": {}, "3_LightGBM기본값": {}}
    for g in GROUP_COLS:
        preds_by_baseline["0_시간x월평균"][g] = baseline_hour_month_mean(g, train_mask, valid_idx)
        preds_by_baseline["1_물리파워커브"][g] = baseline_power_curve(g, cv_suffix, valid_idx)
        preds_by_baseline["2_선형회귀"][g] = baseline_linreg(g, cv_suffix, train_mask, valid_idx)
        pred3, best_iter = baseline_lgbm(g, cv_suffix, train_mask, valid_idx)
        preds_by_baseline["3_LightGBM기본값"][g] = pred3
        lgbm_best_iters[(fold_name, g)] = best_iter

    for baseline_name, group_preds in preds_by_baseline.items():
        pred_df = pd.DataFrame(group_preds, index=valid_idx)
        score, one_minus_nmae, ficr = score_predictions(actual_df, pred_df)
        baseline_results.append({
            "fold": fold_name, "baseline": baseline_name,
            "score": score, "1-NMAE": one_minus_nmae, "FICR": ficr,
        })
    print(f"{fold_name} 완료")

baseline_df = pd.DataFrame(baseline_results)
print(lgbm_best_iters)

c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


A안(2024) 완료


c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


B안 fold1 완료


c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


B안 fold2 완료


c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


B안 fold3 완료
{('A안(2024)', 'kpx_group_1'): 78, ('A안(2024)', 'kpx_group_2'): 335, ('A안(2024)', 'kpx_group_3'): 187, ('B안 fold1', 'kpx_group_1'): 112, ('B안 fold1', 'kpx_group_2'): 129, ('B안 fold1', 'kpx_group_3'): 112, ('B안 fold2', 'kpx_group_1'): 78, ('B안 fold2', 'kpx_group_2'): 335, ('B안 fold2', 'kpx_group_3'): 187, ('B안 fold3', 'kpx_group_1'): 177, ('B안 fold3', 'kpx_group_2'): 251, ('B안 fold3', 'kpx_group_3'): 61}


**확인할 것**: 각 fold가 끝날 때마다 `완료` 메시지가 뜹니다. LightGBM은 4 fold × 3 그룹 = 12번 학습하므로 몇 분 걸릴 수 있습니다. `lgbm_best_iters`에서 조기 종료 라운드 수가 2000(=n_estimators 상한)에 딱 붙어있으면 상한을 늘려야 한다는 신호이니 확인해 주세요.

In [17]:
pivot_score = baseline_df.pivot(index="baseline", columns="fold", values="score").round(4)
b_cols = [c for c in pivot_score.columns if c.startswith("B안")]
pivot_score["B안 평균"] = pivot_score[b_cols].mean(axis=1)
pivot_score["B안 표준편차"] = pivot_score[b_cols].std(axis=1)
display(pivot_score)

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3,B안 평균,B안 표준편차
baseline,,,,,,
0_시간x월평균,0.4334,0.4188,0.4486,0.4183,0.428567,0.017351
1_물리파워커브,0.5825,0.5550,0.5653,0.6006,0.573633,0.023915
2_선형회귀,0.5797,0.5429,0.5614,0.6001,0.568133,0.029188
3_LightGBM기본값,0.5965,0.5697,0.5947,0.6154,0.593267,0.022884


**사전 검증 결과(AI가 미리 같은 로직으로 돌려본 참고용 수치 — 민석님이 실제로 실행하면 이 값과 거의 같아야 정상입니다)**:

| baseline | A안(2024) | B안 fold1 | B안 fold2 | B안 fold3 |
|---|---|---|---|---|
| 0_시간x월평균 | 0.4334 | 0.4188 | 0.4486 | 0.4183 |
| 1_물리파워커브 | 0.5825 | 0.5550 | 0.5653 | 0.6006 |
| 2_선형회귀 | 0.5797 | 0.5429 | 0.5614 | 0.6001 |
| 3_LightGBM기본값 | 0.5965 | 0.5697 | 0.5947 | 0.6154 |

**확인할 것 — 사다리가 꼭 0<1<2<3 순서로 오르는 게 아닙니다**:
- 0 < 1 < 3은 모든 fold에서 뚜렷합니다(기상정보의 가치, 그리고 트리 모델+전체 피처의 가치).
- **2(선형회귀)가 1(물리 파워커브)보다 모든 fold에서 근소하게 낮습니다.** 이건 버그가 아니라 물리적으로 설명되는 결과입니다: 파워커브는 컷인(3m/s)~정격(12m/s)에서 급격히 꺾이고 그 이후엔 평평해지는 S자형 곡선인데, 3차 다항식(추정풍속+제곱+세제곱)으로 전체 구간을 한 번에 피팅하면 이 꺾임(특히 정격 이후 평평해지는 부분)을 잘 못 따라갑니다. 반면 물리 파워커브는 SCADA 실측을 0.5m/s 구간별로 그대로 평균낸 것이라 이 비선형을 그대로 담고 있습니다.
  - **도메인 관점**: 파워커브의 비선형성은 다항식보다 구간별 실측 평균(또는 트리 모델의 분기)이 더 잘 잡는다는, 익히 알려진 사실과 정확히 일치합니다.
  - **DS 관점**: 이 결과는 "피처를 억지로 늘린다고 항상 좋아지는 게 아니다"를 보여주는 좋은 예시이자, 3(LightGBM)이 1·2를 모두 확실히 이기는 것으로 미루어 트리 기반 비선형 모델이 이 문제에 적합하다는 것도 같이 확인해줍니다(model-selection 스킬 2절의 "GBDT 계열이 우세하다"는 가정과 부합).
- 만약 실제 실행 결과가 위 표와 크게 다르면(특히 3이 1·2보다 낮게 나오면) 피처/누수 문제를 의심하고 알려주세요.

In [18]:
display(baseline_df.pivot(index="baseline", columns="fold", values="1-NMAE").round(4))
display(baseline_df.pivot(index="baseline", columns="fold", values="FICR").round(4))

fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
baseline,,,,
0_시간x월평균,0.7487,0.7312,0.7602,0.7366
1_물리파워커브,0.8531,0.8284,0.8472,0.8595
2_선형회귀,0.8601,0.8346,0.8533,0.8675
3_LightGBM기본값,0.8659,0.8475,0.8646,0.8736


fold,A안(2024),B안 fold1,B안 fold2,B안 fold3
baseline,,,,
0_시간x월평균,0.1182,0.1063,0.1370,0.1000
1_물리파워커브,0.3119,0.2817,0.2835,0.3416
2_선형회귀,0.2994,0.2512,0.2695,0.3327
3_LightGBM기본값,0.3271,0.2920,0.3248,0.3572


## 8. 요약 및 다음 단계

**이번 노트북(1~7절)에서 확정한 것**:
- A안+B안 fold 구조와 fold-safe 컬럼 매핑 (`FOLD_SPECS`)
- `ws_est` 파생 6종(sq/cube/corrected/regime_*/high_wind_caution/icing_risk)의 fold-safe 재계산 방식 (`add_fold_safe_ws_features`) — 이번 세션에 새로 발견한 누수 위험을 해결
- 그룹별 피처 프레임 빌더 (`build_group_feature_frame`)와 채점 함수 (`score_predictions`)
- 베이스라인 사다리 4단계(시간×월 평균 / 물리 파워커브 / 선형회귀 / LightGBM 기본값) 실행 결과

**다음 단계 (민석님 실행·확인 후 이어서 작성)**:
1. 위 표를 보고 사다리가 예상대로(0<1<2<3) 올라가는지 확인
2. 확인되면 9절 이후에 **후보 모델 비교**(LightGBM 기본값 vs XGBoost vs CatBoost, 그룹별 3모델 vs 통합 1모델 구조, 타깃 스케일(kWh vs 이용률) 실험)를 추가로 작성
3. `reports/04_model_selection.md`에 이 단계 결과를 Why/How/Result/So-what 구조로 기록

## 9. 후보 모델 비교 — LightGBM / XGBoost / CatBoost (+ 단조 제약) / MLP

**Why**: 7절 베이스라인에서 LightGBM 기본값이 물리 파워커브·선형회귀를 확실히 이겼습니다. 이제 트리 계열 라이브러리 3종(LightGBM/XGBoost/CatBoost)을 공정하게 비교하고, 민석님이 제안하신 두 아이디어를 더합니다.

1. **단조 제약(monotonic constraint)**: "추정풍속이 세지면 예측 발전량도 커지거나 그대로여야 한다"는 물리 법칙(파워커브는 원래 단조증가)을 트리 모델 구조에 직접 강제합니다. `{group}_ws_est_cv_*`, `{group}_power_curve_est_cv_*` 두 컬럼에만 "증가 방향(+1)" 제약을 겁니다 — 나머지 피처(결빙 위험, 돌풍성, 그룹ID 등)는 단조성 근거가 없어 제약을 걸지 않습니다.
2. **MLP(신경망)**: `model-selection` 스킬 원칙("딥러닝은 GBDT 대비 명확한 개선을 보일 때만 도입")에 따라, 지금 GBDT와 나란히 비교해서 실제로 이기는지 확인합니다. 이겨야만 최종 앙상블(9~10단계) 후보로 남습니다.

**공정성 유지**: 7종 모두 같은 `build_group_feature_frame`(fold-safe 피처), 같은 fold, 같은 손실 기준(MAE 계열), 같은 early stopping 방식(학습구간 내부 시간순 마지막 10%)을 씁니다.

### 9-1. 단조 제약 스펙 만들기

**무엇을**: 피처 컬럼 순서에 맞춰 "이 컬럼은 +1(증가 방향 제약), 나머지는 0(제약 없음)"인 리스트를 만듭니다. 라이브러리마다 이 리스트를 넘기는 형식이 달라서(LightGBM/CatBoost는 리스트 그대로, XGBoost는 문자열), 변환은 각 모델 함수 안에서 처리합니다.

In [19]:
def build_monotone_spec(columns, g, cv_suffix):
    """ws_est_cv/power_curve_est_cv 두 컬럼만 +1(비감소) 제약. 나머지는 0(제약 없음)."""
    up_cols = {f"{g}_ws_est_cv_{cv_suffix}", f"{g}_power_curve_est_cv_{cv_suffix}"}
    return [1 if c in up_cols else 0 for c in columns]

### 9-2. GBDT 3종 (LightGBM/XGBoost/CatBoost) 학습 함수 — 단조 제약 켜기/끄기 지원

**작은 함정 하나 발견**: LightGBM은 `monotone_constraints`를 `regression_l1`(MAE)이나 `quantile` 목적함수와 같이 못 씁니다(제약 알고리즘이 2차 도함수를 쓰는데, L1 계열은 2차 도함수가 0이라 라이브러리 자체에서 에러를 냅니다 — 직접 돌려보고 확인함). 그래서 **LightGBM에 단조 제약을 걸 때만 손실을 `huber`로 바꿉니다**(`model-selection` 스킬의 "비교 실험" 손실 목록에 있는 선택지 — MAE와 비슷하게 이상치에 강건하면서도 2차 도함수가 있어 제약과 호환됨). XGBoost·CatBoost는 이런 제약이 없어 MAE 그대로 씁니다. early stopping 기준(`eval_metric`)은 네 경우 모두 MAE로 통일해서, 학습 손실이 달라도 "언제 멈출지" 판단 기준은 공정하게 맞췄습니다.

In [20]:
def fit_predict_gbdt(model_type, g, cv_suffix, train_mask, valid_idx, monotone):
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)
    Xtr, ytr = X_full.loc[tr_mask], train.loc[tr_mask, g]
    Xes, yes = X_full.loc[es_mask], train.loc[es_mask, g]
    mono_spec = build_monotone_spec(X_full.columns, g, cv_suffix) if monotone else None

    if model_type == "lightgbm":
        objective = "huber" if monotone else "l1"  # 단조 제약 + L1 조합은 LightGBM이 지원 안 함
        params = dict(objective=objective, random_state=SEED, n_estimators=2000, verbosity=-1)
        if mono_spec is not None:
            params["monotone_constraints"] = mono_spec
        model = lgb.LGBMRegressor(**params)
        model.fit(Xtr, ytr, eval_set=[(Xes, yes)], eval_metric="l1",
                   callbacks=[lgb.early_stopping(50, verbose=False)])
    elif model_type == "xgboost":
        params = dict(objective="reg:absoluteerror", random_state=SEED, n_estimators=2000,
                      early_stopping_rounds=50, eval_metric="mae")
        if mono_spec is not None:
            params["monotone_constraints"] = "(" + ",".join(map(str, mono_spec)) + ")"
        model = xgb.XGBRegressor(**params)
        model.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False)
    elif model_type == "catboost":
        params = dict(loss_function="MAE", random_state=SEED, n_estimators=2000,
                      early_stopping_rounds=50, verbose=False)
        if mono_spec is not None:
            params["monotone_constraints"] = mono_spec
        model = CatBoostRegressor(**params)
        model.fit(Xtr, ytr, eval_set=(Xes, yes))
    else:
        raise ValueError(model_type)

    pred = pd.Series(model.predict(X_full.loc[valid_idx]), index=valid_idx)
    return pred.clip(lower=0, upper=CAPACITY_KWH[g])

### 9-3. MLP (PyTorch)

**왜 스케일링을 새로 하는가**: 트리 모델(LightGBM 등)은 피처 크기(단위)가 달라도 분기 기준만 찾으면 되니 상관없지만, 신경망은 입력 크기가 들쭉날쭉하면(기온 ~270K, 기압 ~100000Pa, 더미 0/1 등) 학습이 잘 안 됩니다. 그래서 학습 구간의 평균·표준편차로 표준화(z-score)합니다.

**왜 타깃(발전량)을 설비용량으로 나누는가**: 발전량(kWh) 그대로 쓰면 그룹마다 스케일이 달라(21600 vs 21000) 손실 함수 값이 커져서 학습이 불안정해집니다. 0~1 사이 이용률로 바꾸면 신경망 학습이 안정적입니다(예측할 때 다시 설비용량을 곱해 kWh로 되돌립니다). **주의**: 이건 MLP 학습 안정성을 위한 임시 스케일 변환일 뿐, "타깃 스케일(kWh vs 이용률)을 어떤 모델 구조로 최종 채택할지"는 이후 별도 실험(모델 구조 결정)에서 다룹니다.

**구조**: 입력층 → 128 → 64 → 출력 1 (ReLU), Adam 옵티마이저, L1 손실(MAE와 동일 — 다른 후보와 손실 기준 통일), early stopping은 GBDT와 같은 방식(학습구간 내부 시간순 마지막 10%).

In [21]:
import torch
import torch.nn as nn


class SimpleMLP(nn.Module):
    def __init__(self, n_features, hidden=(128, 64)):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def fit_predict_mlp(g, cv_suffix, train_mask, valid_idx, max_epochs=200, patience=20):
    capacity = CAPACITY_KWH[g]
    X_full = build_group_feature_frame(train, g, cv_suffix)
    fit_idx = train_mask & train[g].notna()
    fit_times = train.loc[fit_idx, "kst_dtm"]
    cutoff_es = fit_times.quantile(0.9)
    es_mask = fit_idx & (train["kst_dtm"] > cutoff_es)
    tr_mask = fit_idx & (train["kst_dtm"] <= cutoff_es)

    mean = X_full.loc[tr_mask].mean()
    std = X_full.loc[tr_mask].std().replace(0, 1)

    def to_tensor(df):
        return torch.tensor(((df - mean) / std).to_numpy(), dtype=torch.float32)

    Xtr_t = to_tensor(X_full.loc[tr_mask])
    ytr_t = torch.tensor(train.loc[tr_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xes_t = to_tensor(X_full.loc[es_mask])
    yes_t = torch.tensor(train.loc[es_mask, g].to_numpy() / capacity, dtype=torch.float32)
    Xva_t = to_tensor(X_full.loc[valid_idx])

    torch.manual_seed(SEED)
    model = SimpleMLP(Xtr_t.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.L1Loss()

    best_es, best_state, patience_left = float("inf"), None, patience
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr_t), ytr_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            es_loss = loss_fn(model(Xes_t), yes_t).item()
        if es_loss < best_es - 1e-6:
            best_es, best_state, patience_left = es_loss, {k: v.clone() for k, v in model.state_dict().items()}, patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred_va = model(Xva_t).numpy() * capacity
    return pd.Series(pred_va, index=valid_idx).clip(lower=0, upper=capacity), epoch

### 9-4. 7개 후보(LightGBM/XGBoost/CatBoost × 단조제약 유무 + MLP)를 모든 fold에서 실행

In [ ]:
candidate_results = []
mlp_epochs_used = {}

for fold_name, spec in FOLD_SPECS.items():
    cv_suffix = spec["cv_suffix"]
    train_mask = train["kst_dtm"] < CUTOFFS[cv_suffix]
    valid_mask = (train["kst_dtm"] >= spec["valid_start"]) & (train["kst_dtm"] < spec["valid_end"])
    valid_idx = train.index[valid_mask]
    actual_df = train.loc[valid_idx, TARGET_COLS]

    variant_names = ["lightgbm", "lightgbm_mono", "xgboost", "xgboost_mono", "catboost", "catboost_mono", "mlp"]
    preds_by_variant = {v: {} for v in variant_names}
    for g in GROUP_COLS:
        preds_by_variant["lightgbm"][g] = fit_predict_gbdt("lightgbm", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["lightgbm_mono"][g] = fit_predict_gbdt("lightgbm", g, cv_suffix, train_mask, valid_idx, True)
        preds_by_variant["xgboost"][g] = fit_predict_gbdt("xgboost", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["xgboost_mono"][g] = fit_predict_gbdt("xgboost", g, cv_suffix, train_mask, valid_idx, True)
        preds_by_variant["catboost"][g] = fit_predict_gbdt("catboost", g, cv_suffix, train_mask, valid_idx, False)
        preds_by_variant["catboost_mono"][g] = fit_predict_gbdt("catboost", g, cv_suffix, train_mask, valid_idx, True)
        pred_mlp, n_epoch = fit_predict_mlp(g, cv_suffix, train_mask, valid_idx)
        preds_by_variant["mlp"][g] = pred_mlp
        mlp_epochs_used[(fold_name, g)] = n_epoch

    for variant_name, group_preds in preds_by_variant.items():
        pred_df = pd.DataFrame(group_preds, index=valid_idx)
        score, one_minus_nmae, ficr = score_predictions(actual_df, pred_df)
        candidate_results.append({"fold": fold_name, "model": variant_name, "score": score, "1-NMAE": one_minus_nmae, "FICR": ficr})
    print(f"{fold_name} 완료")

candidate_df = pd.DataFrame(candidate_results)
print(mlp_epochs_used)

c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\cho03\Desktop\wind_forecast_new\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' inste

**확인할 것**: fold마다 7개 후보 × 3그룹 = 21번 학습이 돕니다(GBDT 18번 + MLP 3번). 전체 4 fold라 시간이 꽤 걸릴 수 있습니다(수십 분 단위 각오). `mlp_epochs_used`가 200(=max_epochs 상한)에 자꾸 붙어있으면 상한을 늘려야 한다는 신호입니다.

In [ ]:
pivot_candidate = candidate_df.pivot(index="model", columns="fold", values="score").round(4)
b_cols = [c for c in pivot_candidate.columns if c.startswith("B안")]
pivot_candidate["B안 평균"] = pivot_candidate[b_cols].mean(axis=1)
pivot_candidate["B안 표준편차"] = pivot_candidate[b_cols].std(axis=1)
display(pivot_candidate)

**확인할 것 — 세 가지 질문에 답하기 위한 표입니다**:
1. **어떤 GBDT 라이브러리가 제일 좋은가** (`lightgbm` vs `xgboost` vs `catboost`, 단조 제약 없는 버전끼리 비교)
2. **단조 제약이 도움이 되는가** (`xxx` vs `xxx_mono` 짝을 비교 — A안과 B안 3-fold 모두에서 개선돼야 "효과 있음"으로 채택. 한쪽만 개선되면 `timeseries-validation` 스킬 원칙대로 "효과 불확실")
3. **MLP가 GBDT를 이기는가** (`mlp`가 최고 GBDT 변형보다 낮으면, `model-selection` 스킬 원칙에 따라 지금 단계에서는 후보에서 제외 — 나중에 앙상블 다양성 확보용으로만 재고려)

## 10. 요약 및 다음 단계 (9절 추가 후 갱신)

민석님이 위 결과를 실행·확인해주시면, 이 절에 결론(어떤 모델을 05_tuning으로 넘길지)과 `reports/04_model_selection.md` 작성을 이어가겠습니다.